In [2]:
import os
import requests
from datetime import datetime

# URL context from your first image:
# https://nwdp.nwic.gov.in/dataset_api/resource_list?dataset_id=f1e98026-1cbe-43c1-aba4-9b9d9e679789

# Resource IDs from your second and third images.
RIVER_RESOURCES = {
    "river_water_level_telemetry_2021_2025": "ed8b2d99-a015-40b3-a377-a138969ac371",
    "river_water_level_telemetry_2026_2030": "dbdbeb64-73a7-4056-9623-8cc33bd71001",
}

RESOURCE_SHOW_URL = "https://nwdp.nwic.gov.in/api/3/action/resource_show"
CONNECT_TIMEOUT = 15
READ_TIMEOUT = 45
CHUNK_SIZE = 512 * 1024
PROGRESS_EVERY_BYTES = 5 * 1024 * 1024

def format_size(num_bytes):
    if num_bytes >= 1024 ** 3:
        return f"{num_bytes / (1024 ** 3):.2f} GB"
    if num_bytes >= 1024 ** 2:
        return f"{num_bytes / (1024 ** 2):.2f} MB"
    if num_bytes >= 1024:
        return f"{num_bytes / 1024:.2f} KB"
    return f"{num_bytes} B"

def resolve_download_url(resource_id):
    response = requests.get(
        RESOURCE_SHOW_URL,
        params={"id": resource_id},
        timeout=30,
    )
    response.raise_for_status()

    payload = response.json()
    if not payload.get("success"):
        raise ValueError(f"resource_show failed for resource_id={resource_id}")

    resource = payload.get("result", {})
    download_url = resource.get("url")
    if not download_url:
        raise ValueError(f"No download URL found for resource_id={resource_id}")

    return resource.get("name", "Unknown"), download_url

def download_file(url, output_path):
    temp_path = f"{output_path}.part"
    with requests.get(url, stream=True, timeout=(CONNECT_TIMEOUT, READ_TIMEOUT)) as r:
        r.raise_for_status()
        total_bytes = int(r.headers.get("Content-Length", 0) or 0)
        downloaded = 0
        next_report_at = PROGRESS_EVERY_BYTES

        print(f"Starting download -> {output_path}", flush=True)
        if total_bytes:
            print(f"Expected size -> {format_size(total_bytes)}", flush=True)
        else:
            print("Expected size -> unknown", flush=True)

        with open(temp_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=CHUNK_SIZE):
                if not chunk:
                    continue

                f.write(chunk)
                downloaded += len(chunk)

                if downloaded >= next_report_at or (total_bytes and downloaded == total_bytes):
                    if total_bytes:
                        percent = downloaded / total_bytes * 100
                        print(
                            f"Downloaded {format_size(downloaded)} / {format_size(total_bytes)} ({percent:.1f}%)",
                            flush=True,
                        )
                    else:
                        print(f"Downloaded {format_size(downloaded)}", flush=True)

                    next_report_at = downloaded + PROGRESS_EVERY_BYTES

    os.replace(temp_path, output_path)
    print(f"Download complete -> {output_path}", flush=True)

def download_river_telemetry(resource_name, resource_id):
    try:
        dataset_title, download_url = resolve_download_url(resource_id)
        output_file = f"{resource_name}_{datetime.now().date()}.csv"
        download_file(download_url, output_file)

        print("Dataset:", dataset_title)
        print("Saved ->", output_file)
        print("Source URL ->", download_url)
        print("-" * 70)

    except Exception as e:
        print(f"{resource_name} error: {e}")
        print("-" * 70)

def run():
    for resource_name, resource_id in RIVER_RESOURCES.items():
        download_river_telemetry(resource_name, resource_id)

run()

Starting download -> river_water_level_telemetry_2021_2025_2026-03-14.csv
Expected size -> 310.83 MB
river_water_level_telemetry_2021_2025 error: HTTPSConnectionPool(host='nwdp.nwic.gov.in', port=443): Read timed out.
----------------------------------------------------------------------
Starting download -> river_water_level_telemetry_2026_2030_2026-03-14.csv
Expected size -> 12.47 MB
river_water_level_telemetry_2026_2030 error: HTTPSConnectionPool(host='nwdp.nwic.gov.in', port=443): Read timed out.
----------------------------------------------------------------------


In [7]:
import os
import time
import requests

# River water level telemetry resource IDs from your images.
RIVER_2021_2025_ID = "ed8b2d99-a015-40b3-a377-a138969ac371"
RIVER_2026_2030_ID = "dbdbeb64-73a7-4056-9623-8cc33bd71001"

RESOURCE_SHOW_URL = "https://nwdp.nwic.gov.in/api/3/action/resource_show"
CONNECT_TIMEOUT = 30
READ_TIMEOUT = 900
CHUNK_SIZE = 1024 * 1024
PROGRESS_EVERY_MB = 10
MAX_RETRIES = 8

def format_size(num_bytes):
    if num_bytes >= 1024 ** 3:
        return f"{num_bytes / (1024 ** 3):.2f} GB"
    if num_bytes >= 1024 ** 2:
        return f"{num_bytes / (1024 ** 2):.2f} MB"
    if num_bytes >= 1024:
        return f"{num_bytes / 1024:.2f} KB"
    return f"{num_bytes} B"

def resolve_download_url(resource_id):
    resp = requests.get(
        RESOURCE_SHOW_URL,
        params={"id": resource_id},
        timeout=60,
    )
    resp.raise_for_status()

    payload = resp.json()
    if not payload.get("success"):
        raise ValueError(f"resource_show failed for resource_id={resource_id}")

    result = payload.get("result", {})
    url = result.get("url")
    if not url:
        raise ValueError(f"No download URL found for resource_id={resource_id}")

    return result.get("name", "Unknown"), url

def download_with_resume(url, output_file):
    temp_file = output_file + ".part"
    progress_step = PROGRESS_EVERY_MB * 1024 * 1024

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            existing_bytes = os.path.getsize(temp_file) if os.path.exists(temp_file) else 0
            headers = {"Range": f"bytes={existing_bytes}-"} if existing_bytes > 0 else {}

            with requests.get(
                url,
                headers=headers,
                stream=True,
                timeout=(CONNECT_TIMEOUT, READ_TIMEOUT),
            ) as r:
                if r.status_code == 416 and os.path.exists(temp_file):
                    os.replace(temp_file, output_file)
                    print(f"Already complete -> {output_file}", flush=True)
                    return

                r.raise_for_status()

                if existing_bytes > 0 and r.status_code == 206:
                    mode = "ab"
                    total_bytes = existing_bytes + int(r.headers.get("Content-Length", 0) or 0)
                else:
                    mode = "wb"
                    existing_bytes = 0
                    total_bytes = int(r.headers.get("Content-Length", 0) or 0)

                downloaded = existing_bytes
                next_report = ((downloaded // progress_step) + 1) * progress_step

                print(f"Downloading -> {output_file} (attempt {attempt}/{MAX_RETRIES})", flush=True)
                if total_bytes:
                    print(f"Total size -> {format_size(total_bytes)}", flush=True)
                else:
                    print("Total size -> unknown", flush=True)

                with open(temp_file, mode) as f:
                    for chunk in r.iter_content(chunk_size=CHUNK_SIZE):
                        if not chunk:
                            continue
                        f.write(chunk)
                        downloaded += len(chunk)

                        if downloaded >= next_report:
                            if total_bytes:
                                pct = downloaded / total_bytes * 100
                                print(
                                    f"Progress: {format_size(downloaded)} / {format_size(total_bytes)} ({pct:.1f}%)",
                                    flush=True,
                                )
                            else:
                                print(f"Progress: {format_size(downloaded)}", flush=True)
                            next_report += progress_step

                os.replace(temp_file, output_file)
                print(f"Saved -> {output_file}", flush=True)
                return

        except Exception as e:
            wait_s = min(60, 2 ** attempt)
            print(f"Retry {attempt}/{MAX_RETRIES} for {output_file}: {e}", flush=True)
            if attempt == MAX_RETRIES:
                raise
            time.sleep(wait_s)

def run_river_downloads():
    targets = [
        ("River Water Level Telemetry (2021-2025)", RIVER_2021_2025_ID, "riverwater_level_2021_2025.csv"),
        ("River Water Level Telemetry (2026-2030)", RIVER_2026_2030_ID, "riverwater_level_2026_2030.csv"),
    ]

    for label, resource_id, output_file in targets:
        print("=" * 72)
        print(label)
        try:
            title, download_url = resolve_download_url(resource_id)
            print("Dataset:", title)
            print("Resource ID:", resource_id)
            print("Download URL:", download_url)
            download_with_resume(download_url, output_file)
        except Exception as e:
            print(f"Failed -> {output_file}: {e}", flush=True)

run_river_downloads()

Dataset: River Water Level Chhattisgarh_SW Mahanadi (2021 - 2025) Telemetry Hourly
Source URL: https://nwdp.nwic.gov.in/dataset/f1e98026-1cbe-43c1-aba4-9b9d9e679789/resource/ed8b2d99-a015-40b3-a377-a138969ac371/download/rwl_tel_hr_chhattisgarh_sw_005_2021_2025.csv
Source size: 310.83 MB


Dataset: River Water Level Chhattisgarh_SW Mahanadi (2021 - 2025) Telemetry Hourly
Source URL: https://nwdp.nwic.gov.in/dataset/f1e98026-1cbe-43c1-aba4-9b9d9e679789/resource/ed8b2d99-a015-40b3-a377-a138969ac371/download/rwl_tel_hr_chhattisgarh_sw_005_2021_2025.csv
Source size: 310.83 MB


KeyboardInterrupt: 

In [6]:
import os
import requests
import pandas as pd

# Only this resource (2026-2030), as requested.
RIVER_2026_2030_ID = "dbdbeb64-73a7-4056-9623-8cc33bd71001"
TARGET_DISTRICT = "RAIPUR"
RESOURCE_SHOW_URL = "https://nwdp.nwic.gov.in/api/3/action/resource_show"
OUTPUT_FILE = "riverwater_level_2026_2030_raipur.csv"
PART_FILE = OUTPUT_FILE + ".part"
CHUNKSIZE = 200000

def format_size(num_bytes):
    if num_bytes >= 1024 ** 3:
        return f"{num_bytes / (1024 ** 3):.2f} GB"
    if num_bytes >= 1024 ** 2:
        return f"{num_bytes / (1024 ** 2):.2f} MB"
    if num_bytes >= 1024:
        return f"{num_bytes / 1024:.2f} KB"
    return f"{num_bytes} B"

def resolve_download_url(resource_id):
    resp = requests.get(
        RESOURCE_SHOW_URL,
        params={"id": resource_id},
        timeout=60,
    )
    resp.raise_for_status()

    payload = resp.json()
    if not payload.get("success"):
        raise ValueError(f"resource_show failed for resource_id={resource_id}")

    result = payload.get("result", {})
    download_url = result.get("url")
    if not download_url:
        raise ValueError(f"No download URL found for resource_id={resource_id}")

    return result.get("name", "Unknown"), download_url

def find_district_column(columns):
    for c in columns:
        if str(c).strip().lower() == "district":
            return c
    return None

def filter_raipur_2026_2030_to_part():
    title, url = resolve_download_url(RIVER_2026_2030_ID)
    print("Dataset:", title)
    print("Source URL:", url)

    # Optional size hint from server to monitor transfer scale.
    try:
        head_resp = requests.head(url, allow_redirects=True, timeout=60)
        size_hint = int(head_resp.headers.get("Content-Length", 0) or 0)
        if size_hint:
            print("Source size:", format_size(size_hint))
        else:
            print("Source size: unknown")
    except Exception:
        print("Source size: unknown")

    if os.path.exists(PART_FILE):
        os.remove(PART_FILE)
    if os.path.exists(OUTPUT_FILE):
        os.remove(OUTPUT_FILE)

    total_rows = 0
    matched_rows = 0
    chunk_no = 0
    first_write = True
    district_col = None
    all_columns = None

    # Stream in chunks, filter rows, and write only filtered rows to .part file.
    for chunk in pd.read_csv(url, chunksize=CHUNKSIZE):
        chunk_no += 1
        total_rows += len(chunk)

        if district_col is None:
            district_col = find_district_column(chunk.columns)
            if district_col is None:
                raise ValueError("District column not found in dataset")
            all_columns = list(chunk.columns)

        filtered = chunk[
            chunk[district_col].astype(str).str.strip().str.upper() == TARGET_DISTRICT
        ]

        if not filtered.empty:
            filtered.to_csv(
                PART_FILE,
                mode="w" if first_write else "a",
                index=False,
                header=first_write,
            )
            first_write = False
            matched_rows += len(filtered)

        print(
            f"Chunk {chunk_no}: processed {total_rows:,} rows | matched {matched_rows:,} rows -> {PART_FILE}",
            flush=True,
        )

    if first_write:
        # No Raipur rows found; still create a valid CSV in .part with source headers.
        pd.DataFrame(columns=all_columns if all_columns else []).to_csv(PART_FILE, index=False)
        print("No Raipur records found. Created empty part file:", PART_FILE)

    os.replace(PART_FILE, OUTPUT_FILE)
    print("Final file ready ->", OUTPUT_FILE)
    print(f"Final Raipur rows: {matched_rows:,}")

filter_raipur_2026_2030_to_part()

Dataset: River Water Level Chhattisgarh_SW Mahanadi (2026 - 2030) Telemetry Hourly
Source URL: https://nwdp.nwic.gov.in/dataset/f1e98026-1cbe-43c1-aba4-9b9d9e679789/resource/dbdbeb64-73a7-4056-9623-8cc33bd71001/download/rwl_tel_hr_chhattisgarh_sw_005_2026_2030.csv
Source size: 12.47 MB
Chunk 1: processed 70,391 rows | matched 7,379 rows -> riverwater_level_2026_2030_raipur.csv.part
Final file ready -> riverwater_level_2026_2030_raipur.csv
Final Raipur rows: 7,379
